# Проект: Исследование калибровки вероятностей логистической регрессии

### 🎯 Цель проекта
Выявить, насколько можно доверять предсказанным вероятностям (`predict_proba`) логистической регрессии «из коробки» в условиях сильного дисбаланса классов, и проверить, как калибровка исправляет результат.

In [92]:
import pandas as pd
import numpy as np

df = pd.read_csv('train.csv')
X = df.drop(columns='SeriousDlqin2yrs')
y = df.SeriousDlqin2yrs
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104805 entries, 0 to 104804
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Id                                    104805 non-null  int64  
 1   SeriousDlqin2yrs                      104805 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  104805 non-null  float64
 3   age                                   104805 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  104805 non-null  int64  
 5   DebtRatio                             104805 non-null  float64
 6   MonthlyIncome                         84024 non-null   float64
 7   NumberOfOpenCreditLinesAndLoans       104805 non-null  int64  
 8   NumberOfTimes90DaysLate               104805 non-null  int64  
 9   NumberRealEstateLoansOrLines          104805 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  104805 non-null  int64  
 11  

Данные взял из  https://www.kaggle.com/c/GiveMeSomeCredit

### Описание колонок датасета Give Me Some Credit

| Название переменной | Тип | Описание и перевод |
| :--- | :--- | :--- |
| **SeriousDlqin2yrs** | 0 / 1 | **Целевая переменная:** Была ли просрочка 90+ дней в течение 2 лет (1 — да, 0 — нет). |
| **RevolvingUtilizationOfUnsecuredLines** | % | **Коэффициент использования кредита:** Отношение текущего долга к общему кредитному лимиту. |
| **age** | Int | **Возраст** заемщика в годах. |
| **NumberOfTime30-59DaysPastDueNotWorse** | Int | **Количество просрочек от 30 до 59 дней** за последние 2 года. |
| **DebtRatio** | % | **Коэффициент финансовой нагрузки:** Ежемесячные расходы и платежи, деленные на месячный доход. |
| **MonthlyIncome** | Число | **Ежемесячный доход** заемщика. |
| **NumberOfOpenCreditLinesAndLoans** | Int | **Количество открытых кредитов** (например, карты, автокредиты) на данный момент. |
| **NumberOfTime90DaysLate** | Int | **Количество просрочек на 90 и более дней** за все время. |
| **NumberRealEstateLoansOrLines** | Int | **Количество кредитов под залог недвижимости** и ипотеки. |
| **NumberOfTime60-89DaysPastDueNotWorse** | Int | **Количество просрочек от 60 до 89 дней** за последние 2 года. |
| **NumberOfDependents** | Int | **Количество иждивенцев** в семье заемщика (не включая его самого). |


**MonthlyIncome** и **NumberOfDependents** имеют пропуски, заполню их с помощью медианы.                  

In [93]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [94]:
y.value_counts()

,count
SeriousDlqin2yrs,
0,97855
1,6950


Целевая переменная имеет явный перекос в один из классов, попробую реализовать несколько вариантов и сравню результаты

In [95]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline_base = Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scaler',StandardScaler()),
                     ('model',LogisticRegression(random_state=42))])
pipeline_base.fit(X_train,y_train)

pipeline_with_balanc = Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scaler',StandardScaler()),
                     ('model',LogisticRegression(random_state=42, class_weight='balanced'))])
pipeline_with_balanc.fit(X_train,y_train)


from sklearn.calibration import CalibratedClassifierCV

base_model = LogisticRegression(class_weight='balanced', random_state=42)
pipeline_calib  = Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scaler',StandardScaler()),
                     ('model',CalibratedClassifierCV(estimator=base_model, method='isotonic', cv=3))])
pipeline_calib.fit(X_train,y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 CalibratedClassifierCV(cv=3,
                                        estimator=LogisticRegression(class_weight='balanced',
                                                                     random_state=42),
                                        method='isotonic'))])

In [96]:
result = pd.DataFrame(X_val.copy(), columns=X_val.columns)
result['target'] = y_val

In [97]:
result['pred_base'] = pipeline_base.predict_proba(X_val)[:,1]

result['pred_wb'] = pipeline_with_balanc.predict_proba(X_val)[:,1]

result['pred_cal'] = pipeline_calib.predict_proba(X_val)[:,1]


In [98]:
result[['target',	'pred_base',	'pred_wb',	'pred_cal']].head(10)

,target,pred_base,pred_wb,pred_cal
13800,0,0.058962,0.426293,0.048047
43761,0,0.070258,0.453832,0.050461
21918,0,0.025566,0.246468,0.014126
16903,0,0.120062,0.605101,0.134635
35048,0,0.025132,0.239832,0.012389
92808,0,0.035377,0.281414,0.015388
23696,0,0.069048,0.471777,0.058082
31908,0,0.053767,0.403785,0.039633
51205,0,0.020585,0.189510,0.009916
37586,0,0.113361,0.594738,0.104903


Получил базовые предсказания, проверю, действительно ли результат отражает "вероятность" дефолта.

Для этого необходимо отсортировать **pred_base** по возрастанию, составить 4 "кармама" из равных частей, зачем сравнить среднюю **pred_base** в каждом "кармаме" с соотношением **target==y / count(target)**.

Для этого реализую доп функцию

Если модель хорошо откалибрована, то среднее значение **predict_proba** в каждом кармане должно быть максимально близко к доле реальных единиц (**target**) в этом же кармане.

In [99]:
def cut_buket(df_input, proba_name, name_col="baket"):
    """Позволяет делить вероятности по группам
       Аргументы:
       df_input = датафрейм для добавления столбца групп
       proba_name = название столбца где хранится результат predict_proba
       name_col = название для нового столбца"""
    df = df_input.copy()
    df[name_col] = np.where(df[proba_name] <= 0.25, "0-0.25", df[proba_name])
    df[name_col] = np.where((df[proba_name] > 0.25) & (df[proba_name] <= 0.5), "0.25-0.5", df[name_col])
    df[name_col] = np.where((df[proba_name] > 0.5) & (df[proba_name] <= 0.75), "0.5-0.75", df[name_col])
    df[name_col] = np.where(df[proba_name] > 0.75, "0.75-1", df[name_col])
    return df

In [100]:
baket_base = result[['target','pred_base']].copy()

baket_base = baket_base.sort_values(by = 'pred_base')
baket_base['pred_base'] = baket_base['pred_base'].round(6)

baket_base = cut_buket(baket_base, proba_name="pred_base")

baket_base.groupby('baket')[['pred_base','target']].mean().reset_index()

,baket,pred_base,target
0,0-0.25,0.059896,0.059150
1,0.25-0.5,0.337206,0.503212
2,0.5-0.75,0.615942,0.589928
3,0.75-1,0.815710,0.458333


**Первый** карман - идеально предсказанные значения.

**Второй** - среднее предсказание 33%, но из таргетов каждый второй, модель действует агрессивно занижая риски.

**Третий** -в рамках погрешности практические идеально.

**Четвертый** - средняя вероятность дефолта 81.5% но в реальности практически каждый второй, модель действует слишком осторожно.

Также, поскольку реальный **target** в группе 0.75-1 (45.8%) оказался ниже, чем в группе 0.25-0.5 (50.3%), модель потеряла свойство **монотонности**. Она путает более рискованные объекты с менее рискованными. Вероятно из-за шума в данных или сильного дисбаланса классов.

In [101]:
baket_bw = result[['target','pred_wb']].copy()

baket_bw = baket_bw.sort_values(by = 'pred_wb')
baket_bw['pred_wb'] = baket_bw['pred_wb'].round(6)

baket_bw = cut_buket(baket_bw, proba_name="pred_wb")

baket_bw.groupby('baket')[['pred_wb','target']].mean().reset_index()

,baket,pred_wb,target
0,0-0.25,0.202720,0.012482
1,0.25-0.5,0.371215,0.036761
2,0.5-0.75,0.579347,0.114334
3,0.75-1,0.855143,0.474633


Такой вывод получился, так как модель завысила веса таргетному минорному классу, что позволило сохранить монотонность, риски теперь стабильно и предсказуемо растут **от кармана к карману**.

Очевидно, что в таком случает интерпитировать результат **pred_proba** как "вероятность" нельзя, однако выводы об "уверенности" модели более детально описывают действительность

In [102]:
baket_cal = result[['target','pred_cal']].copy()

baket_cal = baket_cal.sort_values(by = 'pred_cal')
baket_cal['pred_cal'] = baket_cal['pred_cal'].round(6)

baket_cal = cut_buket(baket_cal, proba_name="pred_cal")

baket_cal.groupby('baket')[['pred_cal','target']].mean().reset_index()

,baket,pred_cal,target
0,0-0.25,0.047295,0.047466
1,0.25-0.5,0.344779,0.354296
2,0.5-0.75,0.583134,0.598148


Модель хирургически точно перевела "уверенность" в "вероятность". 4-й карман (0.75-1) полностью схлопнулся, так как после калибровки модель перестала завышать скоры, и в этот диапазон больше не попало ни одной строки.

Однако важно следить за объемом данных: **CalibratedClassifierCV** с **изотонической** регрессией в условиях сильного дисбаланса классов может выдавать искаженные вероятности при слишком сильном дроблении выборки (например, при **cv=5**), так как калибратору банально не хватает редких целевых событий для обучения. Снижение **cv** до **3** дало модели необходимый объем данных для точной настройки.

### Инструмент **CalibratedClassifierCV** действительно справляется с задачей перевода относительной «уверенности» модели в реальную математическую «вероятность», значительно повышая качество оценки рисков и точность прогноза финансовых потерь. В то же время логистическая регрессия «из коробки» (особенно в условиях дисбаланса классов) допускает критические ошибки в калибровке, переоценивая или недооценивая реальную плотность дефолтов.

Возможные причины неточности **predict_proba** в логистической регрессии:
* **Дисбаланс классов**
* **Регуляризация**
* **Мультиколлинеарность и шум**

### 🔍 Ключевые выводы исследования

1. **Модели «из коробки» допускает ошибки:** В условиях дисбаланса логистическая регрессия сильно искажает вероятности. Она теряет монотонность и завышает риски на краях.
2. **Балансировка классов — это компромисс:** Включение `class_weight='balanced'` возвращает модели правильную логику (риски начинают стабильно расти от кармана к карману), но цифры предсказаний окончательно перестают быть реальной вероятностью. Их можно использовать только как «ранг» или «уверенность».
3. **CalibratedClassifierCV решает проблему:** Связка сбалансированной модели и изотонической регрессии (`method='isotonic'`) точно переводит абстрактную уверенность в честную математическую вероятность. Предсказания начинают совпадать с реальностью практически один в один.
4. **Осторожно с объемом данных:** Калибровка очень чувствительна к размеру выборки. Если данных мало, стандартная кросс-валидация `cv=5` слишком сильно дробит данные и ломает результат. Снижение параметра до `cv=3` дает калибратору достаточно примеров для идеальной настройки.


Напоследок проведу тот же эксперимент с датафреймом, в котором классы сбалансированы

In [103]:
from sklearn.datasets import make_classification
import pandas as pd


X, y = make_classification(
    n_samples=10000,
    n_features=6,
    n_informative=4,
    weights=[0.5, 0.5],
    random_state=42)

df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(6)])
df['target'] = y
df.target.value_counts(normalize=True)

,proportion
target,
1,0.5008
0,0.4992


In [104]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipeline_base_test = Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scaler',StandardScaler()),
                     ('model',LogisticRegression(random_state=42))])
pipeline_base_test.fit(X_train,y_train)

base_model_test = LogisticRegression(class_weight='balanced', random_state=42)
pipeline_calib_test  = Pipeline([('imputer',SimpleImputer(strategy='median')),
                     ('scaler',StandardScaler()),
                     ('model',CalibratedClassifierCV(estimator=base_model_test, method='isotonic', cv=3))])
pipeline_calib_test.fit(X_train,y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 CalibratedClassifierCV(cv=3,
                                        estimator=LogisticRegression(class_weight='balanced',
                                                                     random_state=42),
                                        method='isotonic'))])

In [105]:
res = pd.DataFrame(X_test)
res['target'] = y_test

res['pred_base'] = pipeline_base_test.predict_proba(X_test)[:,1]

res['pred_cal'] = pipeline_calib_test.predict_proba(X_test)[:,1]

res[['target','pred_base','pred_cal']].head()

,target,pred_base,pred_cal
0,0,0.176207,0.144589
1,0,0.479989,0.523262
2,1,0.210505,0.207407
3,0,0.486172,0.523262
4,0,0.422213,0.449375


In [106]:
print(cut_buket.__doc__)

Позволяет делить вероятности по группам
       Аргументы:
       df_input = датафрейм для добавления столбца групп
       proba_name = название столбца где хранится результат predict_proba
       name_col = название для нового столбца


In [107]:
base = res[['target','pred_base']].copy()

base = cut_buket(base, proba_name="pred_base")

base.groupby('baket')[['pred_base','target']].mean().reset_index()

,baket,pred_base,target
0,0-0.25,0.135357,0.120579
1,0.25-0.5,0.378165,0.409023
2,0.5-0.75,0.626040,0.589744
3,0.75-1,0.882927,0.874204


При отсутствии дисбаланса классов модель достаточно хорошо оценивает вероятность целевого события.

Вероятно это обусловлено функцией потерь **LogLoss**. Математическое ядро модели(сочетание сигмоиды и Log-Loss) изначально создавалось для поиска непрерывной вероятности.

In [108]:
cl = res[['target','pred_cal']].copy()

cl = cut_buket(cl, proba_name="pred_cal")

cl.groupby('baket')[['pred_cal','target']].mean().reset_index()

,baket,pred_cal,target
0,0-0.25,0.101934,0.104377
1,0.25-0.5,0.381057,0.380454
2,0.5-0.75,0.604771,0.583685
3,0.75-1,0.877146,0.877814


Как и ожидается, калибровка идеально справилась с задачей.